<a href="https://colab.research.google.com/github/qk8015-lgtm/114-2-ProgramingLanguage/blob/main/HW4_PTT_GoogleSheet_RAG%E6%95%B4%E7%90%86%E7%89%88.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# HW4：PTT → Google Sheet → RAG（整理版）

這份 notebook 保留完整流程：

1. 爬取 PTT movie 文章
2. 寫入指定 Google Sheet
3. 從 Google Sheet 讀回資料
4. 建立 FAISS RAG 索引
5. 用 Gemini 根據 PTT 資料回答問題

主要修正：原本設定了 `SHEET_URL`，但實際用 `gc.open(WORKSHEET_NAME)` 開啟試算表，容易打開錯的 Spreadsheet。新版固定使用 `gc.open_by_url(SHEET_URL)`。


In [3]:
# 安裝必要套件
!pip -q install gspread gspread-dataframe faiss-cpu sentence-transformers beautifulsoup4 requests google-generativeai


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.2/18.2 MB 77.9 MB/s eta 0:00:00


In [4]:
import gradio as gr

In [5]:
import re
import time
import uuid
from datetime import datetime
from urllib.parse import urljoin

import requests
import pandas as pd
import numpy as np
from bs4 import BeautifulSoup

import faiss
from sentence_transformers import SentenceTransformer

import google.generativeai as genai
from google.colab import auth, userdata
import gspread
from google.auth import default
from gspread_dataframe import set_with_dataframe, get_as_dataframe


/usr/local/lib/python3.12/dist-packages/google/colab/_import_hooks/_hook_injector.py:55: FutureWarning: 

All support for the `google.generativeai` package has ended. It will no longer be receiving 
updates or bug fixes. Please switch to the `google.genai` package as soon as possible.
See README for more details:

https://github.com/google-gemini/deprecated-generative-ai-python/blob/main/README.md

  loader.exec_module(module)


## 1. 基本設定

請確認 `SHEET_URL` 是你要寫入的 Google Sheet。  
`PTT_WORKSHEET_NAME` 是存放 PTT 原始文章的分頁。


In [6]:
SHEET_URL = "https://docs.google.com/spreadsheets/d/1sa2n02HTV0YGSzoM4XJyjpj4pm5tklrC76p6okxCveY/edit?usp=sharing"
PTT_WORKSHEET_NAME = "ptt_movie_posts"
TIMEZONE_NOTE = "Asia/Taipei"

PTT_HEADER = [
    "post_id", "title", "url", "date", "author", "nrec",
    "created_at", "fetched_at", "content"
]

PTT_MOVIE_INDEX = "https://www.ptt.cc/bbs/movie/index.html"
PTT_COOKIES = {"over18": "1"}
USER_AGENT = "Mozilla/5.0 (compatible; Colab PTT crawler)"


## 2. 連線 Google Sheet

這裡是最重要的修正：使用 `open_by_url(SHEET_URL)`，不要用 worksheet 名稱打開 spreadsheet。


In [7]:
auth.authenticate_user()
creds, _ = default()
gc = gspread.authorize(creds)

# 關鍵修正：直接用網址開啟指定 Google Sheet
sh = gc.open_by_url(SHEET_URL)
print(f"✅ 已開啟試算表：{sh.title}")
print(f"🔗 {SHEET_URL}")


✅ 已開啟試算表：HW1_日常支出速算與分攤 
🔗 https://docs.google.com/spreadsheets/d/1sa2n02HTV0YGSzoM4XJyjpj4pm5tklrC76p6okxCveY/edit?usp=sharing


In [8]:
def ensure_worksheet(spreadsheet, title, header, rows=1000):
    """取得或建立 worksheet，並確保表頭正確。"""
    try:
        ws = spreadsheet.worksheet(title)
    except gspread.WorksheetNotFound:
        ws = spreadsheet.add_worksheet(title=title, rows=str(rows), cols=str(len(header) + 5))
        ws.update([header])
        return ws

    values = ws.get_all_values()
    if not values:
        ws.update([header])
    elif values[0] != header:
        # 保留資料但重建欄位較危險，因此這裡直接清掉並重新建立正確表頭。
        # 若你要保留舊資料，請先備份 Google Sheet。
        ws.clear()
        ws.update([header])
    return ws


def read_sheet_df(ws, header):
    """從 worksheet 讀成 DataFrame，並清掉空列。"""
    df = get_as_dataframe(ws, evaluate_formulas=True, dtype=str).dropna(how="all")
    if df.empty:
        return pd.DataFrame(columns=header)
    df = df.loc[:, [c for c in df.columns if not str(c).startswith("Unnamed")]]
    for col in header:
        if col not in df.columns:
            df[col] = ""
    return df[header].fillna("")


def write_sheet_df(ws, df, header):
    """把 DataFrame 寫回 worksheet。"""
    df_out = df.copy()
    for col in header:
        if col not in df_out.columns:
            df_out[col] = ""
    df_out = df_out[header].infer_objects(copy=False).fillna("") # Add infer_objects to address FutureWarning

    # Google Sheet 寫入前統一轉字串，避免 Timestamp / NaN 型別問題
    for c in df_out.columns:
        df_out[c] = df_out[c].astype(str)

    ws.clear()
    set_with_dataframe(ws, df_out, include_index=False, include_column_header=True, resize=True)
    return len(df_out)


ws_ptt = ensure_worksheet(sh, PTT_WORKSHEET_NAME, PTT_HEADER)
print(f"✅ 已準備 worksheet：{ws_ptt.title}")

✅ 已準備 worksheet：ptt_movie_posts


## 3. PTT movie 爬蟲

這段只負責爬 PTT，不碰 RAG。資料會先存在 `new_posts_df`。


In [9]:
def now_iso():
    return datetime.now().isoformat(timespec="seconds")


def get_soup(url):
    resp = requests.get(
        url,
        timeout=20,
        headers={"User-Agent": USER_AGENT},
        cookies=PTT_COOKIES,
    )
    resp.raise_for_status()
    return BeautifulSoup(resp.text, "html.parser")


def get_prev_index_url(soup):
    for a in soup.select("div.btn-group-paging a.btn.wide"):
        if "上頁" in a.get_text(strip=True):
            href = a.get("href")
            return urljoin("https://www.ptt.cc", href) if href else None
    return None


def parse_nrec(nrec_span):
    if not nrec_span:
        return 0
    txt = nrec_span.get_text(strip=True)
    if txt == "爆":
        return 100
    if txt.startswith("X"):
        try:
            return -int(txt[1:])
        except Exception:
            return -10
    try:
        return int(txt)
    except Exception:
        return 0


def extract_post_list(index_soup):
    posts = []
    for item in index_soup.select("div.r-ent"):
        a = item.select_one("div.title a")
        if not a:
            continue

        title = a.get_text(strip=True)
        url = urljoin("https://www.ptt.cc", a.get("href"))
        author_node = item.select_one("div.author")
        date_node = item.select_one("div.date")
        nrec_node = item.select_one("div.nrec span")

        posts.append({
            "title": title,
            "url": url,
            "author": author_node.get_text(strip=True) if author_node else "",
            "date": date_node.get_text(strip=True) if date_node else "",
            "nrec": parse_nrec(nrec_node),
        })
    return posts


def clean_ptt_content(article_soup):
    main = article_soup.select_one("#main-content")
    if not main:
        return "", ""

    # 取出文章建立時間
    created_at = ""
    metalines = main.select("div.article-metaline")
    for m in metalines:
        tag = m.select_one("span.article-meta-tag")
        value = m.select_one("span.article-meta-value")
        if tag and value and tag.get_text(strip=True) == "時間":
            created_at = value.get_text(strip=True)

    # 移除 meta 與推文
    for node in main.select("div.article-metaline, div.article-metaline-right, div.push"):
        node.decompose()

    text = main.get_text("\n", strip=True)
    text = re.split(r"\n--\n|\n※ 發信站:", text)[0].strip()
    text = re.sub(r"\n{3,}", "\n\n", text)
    return text, created_at


def make_post_id(url):
    # 用文章網址檔名當 post_id，穩定且方便去重
    return url.rstrip("/").split("/")[-1].replace(".html", "")


def crawl_ptt_movie(pages=2, delay=0.5):
    """爬取 PTT movie 最新 pages 頁文章。"""
    all_rows = []
    index_url = PTT_MOVIE_INDEX

    for page in range(int(pages)):
        print(f"📄 正在讀取列表頁 {page + 1}/{pages}: {index_url}")
        index_soup = get_soup(index_url)
        post_list = extract_post_list(index_soup)

        for p in post_list:
            try:
                article_soup = get_soup(p["url"])
                content, created_at = clean_ptt_content(article_soup)
                row = {
                    "post_id": make_post_id(p["url"]),
                    "title": p["title"],
                    "url": p["url"],
                    "date": p["date"],
                    "author": p["author"],
                    "nrec": p["nrec"],
                    "created_at": created_at,
                    "fetched_at": now_iso(),
                    "content": content,
                }
                all_rows.append(row)
                time.sleep(delay)
            except Exception as e:
                print(f"⚠️ 跳過文章：{p.get('title', '')}，原因：{e}")

        prev_url = get_prev_index_url(index_soup)
        if not prev_url:
            break
        index_url = prev_url
        time.sleep(delay)

    df = pd.DataFrame(all_rows, columns=PTT_HEADER)
    print(f"✅ 本次爬到 {len(df)} 篇文章")
    return df

## 4. 執行爬蟲並寫入 Google Sheet

這一格會：

1. 從 Google Sheet 讀取既有資料
2. 爬取新的 PTT 資料
3. 合併並用 `post_id` 去重
4. 寫回 Google Sheet
5. 再讀一次確認真的寫入成功


In [10]:
# 你可以調整 pages，例如 pages=1 先測試，確認成功後再改成 3 或 5
new_posts_df = crawl_ptt_movie(pages=2, delay=1.0)

old_posts_df = read_sheet_df(ws_ptt, PTT_HEADER)
print(f"📌 Google Sheet 原本有 {len(old_posts_df)} 筆")

ptt_posts_df = pd.concat([old_posts_df, new_posts_df], ignore_index=True)
ptt_posts_df = ptt_posts_df.drop_duplicates(subset=["post_id"], keep="last")
ptt_posts_df = ptt_posts_df.sort_values(by="fetched_at", ascending=False)

written_count = write_sheet_df(ws_ptt, ptt_posts_df, PTT_HEADER)
print(f"✅ 已寫入 Google Sheet：{written_count} 筆")

verify_df = read_sheet_df(ws_ptt, PTT_HEADER)
print(f"🔍 從 Google Sheet 重新讀回：{len(verify_df)} 筆")

if len(verify_df) == written_count:
    print("✅ 寫入驗證成功")
else:
    print("⚠️ 寫入筆數與讀回筆數不同，請檢查 Google Sheet 權限或資料格式")

📄 正在讀取列表頁 1/2: https://www.ptt.cc/bbs/movie/index.html
📄 正在讀取列表頁 2/2: https://www.ptt.cc/bbs/movie/index11002.html
⚠️ 跳過文章：[ 好雷]  太空超人 原作粉表示滿足，原因：('Connection aborted.', RemoteDisconnected('Remote end closed connection without response'))
⚠️ 跳過文章：Fw: [情報]金銀島又重拍！休傑克曼主演 雷利史考特導，原因：('Connection aborted.', RemoteDisconnected('Remote end closed connection without response'))
✅ 本次爬到 22 篇文章
📌 Google Sheet 原本有 0 筆
✅ 已寫入 Google Sheet：22 筆
🔍 從 Google Sheet 重新讀回：22 筆
✅ 寫入驗證成功


## 5. 從 Google Sheet 建立 RAG 索引

重點：RAG 不直接吃剛爬下來的記憶體資料，而是**從 Google Sheet 重新讀回**，這樣才能確認流程真的是：

`PTT → Google Sheet → RAG`


In [11]:
# 從 Google Sheet 重新讀取，作為 RAG 的唯一資料來源
rag_source_df = read_sheet_df(ws_ptt, PTT_HEADER)

# 清掉沒有內容的文章
rag_source_df = rag_source_df[rag_source_df["content"].astype(str).str.strip() != ""].copy()
print(f"📚 可用於 RAG 的文章數：{len(rag_source_df)}")

rag_source_df.head()


📚 可用於 RAG 的文章數：22


,post_id,title,url,date,author,nrec,created_at,fetched_at,content
0,M.1781028940.A.1EC,[請益] 末代皇帝,https://www.ptt.cc/bbs/movie/M.1781028940.A.1E...,6/10,shesheder,3,Wed Jun 10 02:15:38 2026,2026-06-10 3:46:27,遇到一個現象\n\n很感好奇\n\n來板上請教各位喜歡電影的板友\n\n我前兩天到光點華山去...
1,M.1781024542.A.7BC,[新聞] 克林伊斯威特 疑似正式退休,https://www.ptt.cc/bbs/movie/M.1781024542.A.7B...,6/10,flysonics,23,Wed Jun 10 01:02:17 2026,2026-06-10 3:46:26,<此篇翻譯名利場報導>\n\n本週，維基百科的編輯們不得不對克林·伊斯威特的頁面進行重大修改...
2,M.1781023166.A.54F,[討論] 《揭秘日》爛番茄解禁,https://www.ptt.cc/bbs/movie/M.1781023166.A.54...,6/10,chyx741021,19,Wed Jun 10 00:39:22 2026,2026-06-10 3:46:24,https://www.rottentomatoes.com/m/disclosure_da...
3,M.1781021511.A.4D2,[討論] 李連杰幫忙宣傳火遮眼,https://www.ptt.cc/bbs/movie/M.1781021511.A.4D...,6/10,godgod777,3,Wed Jun 10 00:11:49 2026,2026-06-10 3:46:23,https://youtu.be/91X4eNrykTs?si=3VpBR7Lvy5L8T1...
4,M.1781011881.A.887,[新聞] 《玩命關頭》導演操刀《絕地戰兵》真人電,https://www.ptt.cc/bbs/movie/M.1781011881.A.88...,06/09,akila08539,2,Tue Jun 9 21:31:18 2026,2026-06-10 3:46:21,新聞網址：\nhttps://www.toy-people.com/?p=110914\n《...


In [12]:
print("正在載入多語言 Embedding 模型...")
embedding_model = SentenceTransformer("paraphrase-multilingual-MiniLM-L12-v2")
print("✅ Embedding 模型載入完成")


def build_rag_documents(df):
    docs = []
    for _, row in df.iterrows():
        title = str(row.get("title", ""))
        content = str(row.get("content", ""))
        url = str(row.get("url", ""))
        author = str(row.get("author", ""))
        date = str(row.get("date", ""))
        nrec = str(row.get("nrec", ""))

        text = (f"標題：{title}\n"
                f"作者：{author}\n"
                f"日期：{date}\n"
                f"推文數：{nrec}\n"
                f"內容：{content}")
        docs.append({
            "post_id": str(row.get("post_id", "")),
            "title": title,
            "url": url,
            "text": text,
        })
    return docs


def build_faiss_index(docs):
    if not docs:
        raise ValueError("沒有可建立索引的文件")

    texts = [d["text"] for d in docs]
    embeddings = embedding_model.encode(texts, convert_to_numpy=True, show_progress_bar=True)
    embeddings = embeddings.astype("float32")

    index = faiss.IndexFlatL2(embeddings.shape[1])
    index.add(embeddings)
    return index, embeddings


rag_documents = build_rag_documents(rag_source_df)
rag_index, rag_embeddings = build_faiss_index(rag_documents)

print(f"✅ RAG 索引建立完成：{len(rag_documents)} 篇文章，向量維度 {rag_embeddings.shape[1]}")

正在載入多語言 Embedding 模型...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:112: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/229 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/122 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/3.89k [00:00<?, ?B/s]

...

special_tokens_map.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

✅ Embedding 模型載入完成


Batches:   0%|          | 0/1 [00:00<?, ?it/s]

✅ RAG 索引建立完成：22 篇文章，向量維度 384


## 6. Gemini 設定與 RAG 問答

請先在 Colab Secrets 裡建立 `gemini`，內容是你的 Gemini API key。


In [22]:
api_key = userdata.get("gemini")
if not api_key:
    raise ValueError("找不到 Colab Secret：gemini。請先在 Colab Secrets 新增 Gemini API key。")

genai.configure(api_key=api_key)

# 修正為正確且存在的模型名稱
GEMINI_MODEL_NAME = "gemini-1.5-flash"
llm = genai.GenerativeModel(GEMINI_MODEL_NAME)
print(f"✅ Gemini 模型已修正為：{GEMINI_MODEL_NAME}")

✅ Gemini 模型已修正為：gemini-1.5-flash


In [14]:
def retrieve_docs(query, k=3):
    if rag_index is None or not rag_documents:
        return []
    q_emb = embedding_model.encode([query], convert_to_numpy=True).astype("float32")
    distances, indices = rag_index.search(q_emb, int(k))

    results = []
    for dist, idx in zip(distances[0], indices[0]):
        if idx == -1:
            continue
        doc = rag_documents[idx].copy()
        doc["distance"] = float(dist)
        results.append(doc)
    return results


def query_rag(question, k=3):
    docs = retrieve_docs(question, k=k)
    if not docs:
        return "找不到相關 PTT 資料。"

    context = "\n\n---\n\n".join(
        [f"來源標題：{d['title']}\n來源網址：{d['url']}\n{d['text']}" for d in docs]
    )

    prompt = f"""
你是一個根據 PTT 電影版資料回答問題的助教。
請只根據【PTT 資料】回答。
如果資料不足，請明確說「目前資料不足，無法判斷」，不要自行編造。
回答請使用繁體中文，並在最後列出參考來源標題與網址。

【PTT 資料】
{context}

【問題】
{question}

【回答】
""".strip()

    response = llm.generate_content(prompt)
    return response.text

## 7. 快速測試


## 8. 使用 Gradio 建立網頁介面

In [25]:
import gradio as gr

def respond(message, chat_history, k_value):
    try:
        if not message.strip():
            return "", chat_history

        # 呼叫 RAG 檢索與 Gemini 回答
        bot_message = query_rag(message, k=int(k_value))

        # 更新對話紀錄
        chat_history.append((message, bot_message))
        return "", chat_history
    except Exception as e:
        error_msg = f"發生錯誤：{str(e)}"
        chat_history.append((message, error_msg))
        return "", chat_history

# 建立 Blocks 介面
with gr.Blocks(theme=gr.themes.Soft(), title="PTT 電影版 AI 助手") as demo:
    gr.Markdown("# 🎬 PTT 電影版 RAG 問答機器人")
    gr.Markdown("此助手會根據 Google Sheet 中的 PTT 資料回答問題。")

    chatbot = gr.Chatbot(label="對話歷史")

    with gr.Row():
        msg = gr.Textbox(
            label="輸入問題",
            placeholder="例如：最近有什麼好片？",
            scale=4
        )
        k_slider = gr.Slider(
            minimum=1,
            maximum=10,
            value=3,
            step=1,
            label="參考文章數 (k)",
            scale=1
        )

    with gr.Row():
        submit_btn = gr.Button("送出", variant="primary")
        clear = gr.Button("清除對話")

    # 設定元件動作
    submit_btn.click(respond, [msg, chatbot, k_slider], [msg, chatbot])
    msg.submit(respond, [msg, chatbot, k_slider], [msg, chatbot])
    clear.click(lambda: None, None, chatbot, queue=False)

    gr.Examples(
        examples=["最近有什麼熱門好片？", "這兩天有哪些雷文？", "介紹一下《末代皇帝》的討論內容"],
        inputs=msg
    )

# 啟動 Gradio，debug=True 有助於在 Colab 直接看到報錯內容
demo.launch(share=True, debug=True, inline=False)

/tmp/ipykernel_471/1407394784.py:20: DeprecationWarning: The 'theme' parameter in the Blocks constructor will be removed in Gradio 6.0. You will need to pass 'theme' to Blocks.launch() instead.
  with gr.Blocks(theme=gr.themes.Soft(), title="PTT 電影版 AI 助手") as demo:
/tmp/ipykernel_471/1407394784.py:24: UserWarning: You have not specified a value for the `type` parameter. Defaulting to the 'tuples' format for chatbot messages, but this is deprecated and will be removed in a future version of Gradio. Please set type='messages' instead, which uses openai-style dictionaries with 'role' and 'content' keys.
  chatbot = gr.Chatbot(label="對話歷史")
/tmp/ipykernel_471/1407394784.py:24: DeprecationWarning: The default value of 'allow_tags' in gr.Chatbot will be changed from False to True in Gradio 6.0. You will need to explicitly set allow_tags=False if you want to disable tags in your chatbot.
  chatbot = gr.Chatbot(label="對話歷史")


Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
* Running on public URL: https://6b07721ae7d2ee6f44.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
Keyboard interruption in main thread... closing server.
Killing tunnel 127.0.0.1:7862 <> https://6b07721ae7d2ee6f44.gradio.live


In [19]:
import gradio as gr

def respond(message, chat_history, k_value):
    """對話式的 RAG 回答函式，加入 k 值調整與錯誤處理"""
    try:
        if not message.strip():
            return "", chat_history + [[message, "請輸入問題！"]]

        # 根據使用者選取的 k 值進行檢索
        bot_message = query_rag(message, k=int(k_value))

        # 更新對話紀錄 (ChatInterface 在 Blocks 中需要手動處理或使用特殊元件)
        chat_history.append((message, bot_message))
        return "", chat_history
    except Exception as e:
        error_msg = f"發生錯誤：{str(e)}"
        chat_history.append((message, error_msg))
        return "", chat_history

# 使用 Blocks 建立更彈性的介面
with gr.Blocks(theme=gr.themes.Soft(), title="PTT 電影版 AI 助手") as demo:
    gr.Markdown("# 🎬 PTT 電影版 RAG 問答機器人")

    with gr.Row():
        with gr.Column(scale=4):
            chatbot = gr.Chatbot(label="對話歷史")
            msg = gr.Textbox(label="輸入問題", placeholder="例如：最近有什麼好片？")
        with gr.Column(scale=1):
            k_slider = gr.Slider(minimum=1, maximum=10, value=3, step=1, label="參考文章數量 (k)", info="較小的值速度較快")
            clear = gr.Button("清除對話")

    # 設定觸發動作
    msg.submit(respond, [msg, chatbot, k_slider], [msg, chatbot])
    clear.click(lambda: None, None, chatbot, queue=False)

    gr.Examples(
        examples=["最近有什麼熱門好片？", "這兩天有哪些雷文？", "推薦哪部動作片？"],
        inputs=msg
    )

# 啟動介面
demo.launch(share=True, debug=True)

/tmp/ipykernel_471/4155095508.py:21: DeprecationWarning: The 'theme' parameter in the Blocks constructor will be removed in Gradio 6.0. You will need to pass 'theme' to Blocks.launch() instead.
  with gr.Blocks(theme=gr.themes.Soft(), title="PTT 電影版 AI 助手") as demo:
/tmp/ipykernel_471/4155095508.py:26: UserWarning: You have not specified a value for the `type` parameter. Defaulting to the 'tuples' format for chatbot messages, but this is deprecated and will be removed in a future version of Gradio. Please set type='messages' instead, which uses openai-style dictionaries with 'role' and 'content' keys.
  chatbot = gr.Chatbot(label="對話歷史")
/tmp/ipykernel_471/4155095508.py:26: DeprecationWarning: The default value of 'allow_tags' in gr.Chatbot will be changed from False to True in Gradio 6.0. You will need to explicitly set allow_tags=False if you want to disable tags in your chatbot.
  chatbot = gr.Chatbot(label="對話歷史")


Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
* Running on public URL: https://f15c23c8dcb5a0cb24.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/gradio/queueing.py", line 759, in process_events
    response = await route_utils.call_process_api(
               ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/gradio/route_utils.py", line 354, in call_process_api
    output = await app.get_blocks().process_api(
             ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/gradio/blocks.py", line 2191, in process_api
    result = await self.call_function(
             ^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/gradio/blocks.py", line 1698, in call_function
    prediction = await anyio.to_thread.run_sync(  # type: ignore
                 ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/anyio/to_thread.py", line 63, in run_sync
    return await get_async_backend().run_sync_in_worker_thread(
           ^^^^^

Keyboard interruption in main thread... closing server.
Killing tunnel 127.0.0.1:7860 <> https://fc4b6b39659b4b81f2.gradio.live
Killing tunnel 127.0.0.1:7861 <> https://db1ecebe36b3f27328.gradio.live
Killing tunnel 127.0.0.1:7862 <> https://f15c23c8dcb5a0cb24.gradio.live


In [ ]:
import gradio as gr

def respond(message, chat_history, k_value):
    try:
        if not message.strip():
            return "", chat_history

        # 調用 RAG 檢索與回答
        bot_message = query_rag(message, k=int(k_value))

        chat_history.append((message, bot_message))
        return "", chat_history
    except Exception as e:
        error_msg = f"發生錯誤：{str(e)}"
        chat_history.append((message, error_msg))
        return "", chat_history

with gr.Blocks(theme=gr.themes.Soft(), title="PTT 電影版 AI 助手") as demo:
    gr.Markdown("# 🎬 PTT 電影版 RAG 問答機器人")

    chatbot = gr.Chatbot(label="對話歷史")
    with gr.Row():
        msg = gr.Textbox(label="輸入問題", placeholder="例如：最近有什麼好片？", scale=4)
        k_slider = gr.Slider(minimum=1, maximum=10, value=3, step=1, label="參考文章數 (k)", scale=1)

    with gr.Row():
        submit_btn = gr.Button("送出", variant="primary")
        clear = gr.Button("清除對話")

    # 設定觸發動作
    submit_btn.click(respond, [msg, chatbot, k_slider], [msg, chatbot])
    msg.submit(respond, [msg, chatbot, k_slider], [msg, chatbot])
    clear.click(lambda: None, None, chatbot, queue=False)

    gr.Examples(
        examples=["最近有什麼熱門好片？", "這兩天有哪些雷文？", "介紹一下《末代皇帝》在討論什麼？"],
        inputs=msg
    )

# 重新啟動介面
demo.launch(share=True, debug=True)

/tmp/ipykernel_471/108213756.py:18: DeprecationWarning: The 'theme' parameter in the Blocks constructor will be removed in Gradio 6.0. You will need to pass 'theme' to Blocks.launch() instead.
  with gr.Blocks(theme=gr.themes.Soft(), title="PTT 電影版 AI 助手") as demo:
/tmp/ipykernel_471/108213756.py:21: UserWarning: You have not specified a value for the `type` parameter. Defaulting to the 'tuples' format for chatbot messages, but this is deprecated and will be removed in a future version of Gradio. Please set type='messages' instead, which uses openai-style dictionaries with 'role' and 'content' keys.
  chatbot = gr.Chatbot(label="對話歷史")
/tmp/ipykernel_471/108213756.py:21: DeprecationWarning: The default value of 'allow_tags' in gr.Chatbot will be changed from False to True in Gradio 6.0. You will need to explicitly set allow_tags=False if you want to disable tags in your chatbot.
  chatbot = gr.Chatbot(label="對話歷史")


Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
* Running on public URL: https://03fed51c9947f1eaa2.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


In [ ]:
import gradio as gr

def respond(message, chat_history, k_value):
    try:
        if not message.strip():
            return "", chat_history

        # 確保調用的是修正過的 query_rag
        bot_message = query_rag(message, k=int(k_value))

        chat_history.append((message, bot_message))
        return "", chat_history
    except Exception as e:
        error_msg = f"發生錯誤：{str(e)}"
        chat_history.append((message, error_msg))
        return "", chat_history

with gr.Blocks(theme=gr.themes.Soft(), title="PTT 電影版 AI 助手") as demo:
    gr.Markdown("# 🎬 PTT 電影版 RAG 問答機器人")
    gr.Markdown("目前模型：`gemini-1.5-flash` | 資料來源：Google Sheet")

    chatbot = gr.Chatbot(label="對話歷史")
    with gr.Row():
        msg = gr.Textbox(label="輸入問題", placeholder="例如：最近有什麼好片？", scale=4)
        k_slider = gr.Slider(minimum=1, maximum=10, value=3, step=1, label="參考文章數 (k)", scale=1)

    with gr.Row():
        submit_btn = gr.Button("送出", variant="primary")
        clear = gr.Button("清除對話")

    submit_btn.click(respond, [msg, chatbot, k_slider], [msg, chatbot])
    msg.submit(respond, [msg, chatbot, k_slider], [msg, chatbot])
    clear.click(lambda: None, None, chatbot, queue=False)

    gr.Examples(
        examples=["最近有什麼熱門好片？", "這兩天有哪些雷文？", "介紹一下《末代皇帝》在討論什麼？"],
        inputs=msg
    )

# 使用 inline=False 確保在 Colab 彈出獨立連結，避免介面卡死
demo.launch(share=True, debug=True, inline=False)

In [ ]:
import gradio as gr

def respond(message, chat_history, k_value):
    try:
        if not message.strip():
            return "", chat_history

        # 執行 RAG 查詢
        bot_message = query_rag(message, k=int(k_value))

        if not bot_message:
            bot_message = "模型未回傳任何內容，請再試一次。"

        chat_history.append((message, bot_message))
        return "", chat_history
    except Exception as e:
        error_msg = f"系統回應逾時或發生錯誤：{str(e)}\n請嘗試減少 k 值或稍後再試。"
        chat_history.append((message, error_msg))
        return "", chat_history

with gr.Blocks(theme=gr.themes.Soft(), title="PTT 電影版 AI 助手") as demo:
    gr.Markdown("# 🎬 PTT 電影版 RAG 問答機器人")
    gr.Markdown("目前狀態：✅ 已連線至 Google Sheet | 🤖 模型：`gemini-1.5-flash`")

    chatbot = gr.Chatbot(label="對話歷史")
    with gr.Row():
        msg = gr.Textbox(label="輸入問題", placeholder="例如：最近有什麼好片？", scale=4)
        k_slider = gr.Slider(minimum=1, maximum=5, value=3, step=1, label="參考文章數 (k)", info="若回報逾時請調低此值", scale=1)

    with gr.Row():
        submit_btn = gr.Button("送出", variant="primary")
        clear = gr.Button("清除對話")

    submit_btn.click(respond, [msg, chatbot, k_slider], [msg, chatbot])
    msg.submit(respond, [msg, chatbot, k_slider], [msg, chatbot])
    clear.click(lambda: None, None, chatbot, queue=False)

# 使用 inline=False 避免 Colab 內部顯示問題，強制產生外部連結
demo.launch(share=True, debug=True, inline=False)

In [ ]:
question = input("請輸入問題：")
answer = query_rag(question, k=3)
print(answer)


## 常見錯誤檢查

如果 PTT 資料沒有寫回 Google Sheet，請依序檢查：

1. 是否有成功印出 `已開啟試算表`，且名稱正確。
2. `SHEET_URL` 是否是你要寫入的那一份 Google Sheet。
3. Google Sheet 權限是否允許目前 Colab 登入的 Google 帳號編輯。
4. 是否執行到「執行爬蟲並寫入 Google Sheet」那一格。
5. 是否有看到 `寫入驗證成功`。
6. RAG 要從 `rag_source_df = read_sheet_df(...)` 開始，確保資料來源是 Google Sheet，而不是記憶體中的暫存變數。
